In [1]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
import pandas as pd
import numpy as np
from scipy import stats

print("Loading model, please wait...")

llm = OllamaLLM(model="gpt-oss:20b")  # 你的模型

# Prompt 版本 A：简单版
prompt_a = PromptTemplate.from_template("""
You are a statistics expert.
Answer the question clearly and concisely.

Question: {question}

Answer:
""")

# Prompt 版本 B：加 few-shot + chain-of-thought
prompt_b = PromptTemplate.from_template("""
You are an experienced statistics professor.
Think step by step before answering.

Example:
Q: What is p-value?
A: p-value is the probability of observing the data (or more extreme) assuming H0 is true.

Now answer this question carefully:

Question: {question}

Answer:
""")

chain_a = prompt_a | llm
chain_b = prompt_b | llm

question = "解释 p-value 在假设检验中的含义，并说明为什么 p < 0.05 不等于实际效果很大。举一个实际例子。"

# 跑 10 次，收集回答长度
n_runs = 10
results = []

print("Running A/B testing...")

for i in range(n_runs):
    print(f"Run {i+1}/{n_runs}")
    
    # Prompt A
    resp_a = chain_a.invoke({"question": question})
    len_a = len(resp_a)
    
    # Prompt B
    resp_b = chain_b.invoke({"question": question})
    len_b = len(resp_b)
    
    results.append({
        "run": i+1,
        "prompt_a_length": len_a,
        "prompt_b_length": len_b
    })

# 转成 DataFrame
df = pd.DataFrame(results)

print("\nResults:")
print(df)

# 统计比较
mean_a = df["prompt_a_length"].mean()
mean_b = df["prompt_b_length"].mean()
t_stat, p_value = stats.ttest_rel(df["prompt_a_length"], df["prompt_b_length"])

print(f"\nMean length Prompt A: {mean_a:.1f}")
print(f"Mean length Prompt B: {mean_b:.1f}")
print(f"p-value (paired t-test): {p_value:.4f}")
print(f"Conclusion: {'Prompt B is significantly longer' if p_value < 0.05 else 'No significant difference'}")

C:\Users\hyc98\anaconda4\envs\llm_eval\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loading model, please wait...
Running A/B testing...
Run 1/10
Run 2/10
Run 3/10
Run 4/10
Run 5/10
Run 6/10
Run 7/10
Run 8/10
Run 9/10
Run 10/10

Results:
   run  prompt_a_length  prompt_b_length
0    1              664             1783
1    2              697             1241
2    3              759             1590
3    4              483             1169
4    5              724             1768
5    6              748              772
6    7              942             1318
7    8              641             1823
8    9              771             1795
9   10              855             1214

Mean length Prompt A: 728.4
Mean length Prompt B: 1447.3
p-value (paired t-test): 0.0002
Conclusion: Prompt B is significantly longer
